# CKA-RL on Meta-World — Kaggle runner

**Panel settings:** Accelerator = **GPU T4 x2**, Internet = **On**, then
`Save Version` → **Save & Run All (Commit)** and close the tab. Kaggle runs it in
the background; limits are **12 h per GPU session** and **20 GB** of auto-saved
`/kaggle/working`.

**Run ONE stage per commit.** Every idea is its own stage, so nothing has to fit
the 12 h wall and stages split freely across two accounts.

| order | STAGE | who | roughly | what it answers |
|---|---|---|---|---|
| 1 | `smoke` | A | ~10 min | does the pipeline run at all |
| 2 | `pilot` | A | ~1 h | do these tasks learn in 150k steps |
| 3 | `baselines s0` | both | ~1.5 h | FT denominators |
| 4 | `cond s0 1` / `cond s0 2` | A | bulk | classic_cka, ± distillation |
| 4 | `cond s0 3` / `cond s0 4` | B | bulk | weight_delta, ± distillation |
| 5 | `pretrain` → `baselines s4` → `cond s4 N` | day 2 | optional | shared-layer idea |
| 6 | `report s0` | one account | minutes | after merging outputs |

**Do not skip `smoke` and `pilot`.** Smoke catches wiring bugs in ten minutes
instead of after an hour. Pilot decides whether the 150k budget is viable at all —
if the tasks do not learn, every later comparison is comparing noise.

## 1. Config — the only cell you normally edit

In [11]:
REPO_URL    = "https://github.com/Yasamin-Rajabi/Continual-RL-Project.git"
REPO_BRANCH = "master"
REPO_SUBDIR = "metaworld"     # folder inside the repo; "" if the code is at the root

STAGE = "smoke"               # smoke | pilot | baselines | cond | report | pretrain | setup | sanity
ARG1  = "s0"                  # for baselines/cond/report: s0 or s4
ARG2  = ""                    # for cond: "1".."4"

# Stop cleanly before Kaggle's 12 h wall so the version still COMMITS and the
# output is saved. A version killed by the wall is marked failed.
BUDGET_HOURS = 10.5

import os, pathlib
WORK = pathlib.Path("/kaggle/working")
CODE = pathlib.Path("/kaggle/temp/repo")   # scratch: not part of the saved output
print("stage:", STAGE, ARG1, ARG2, "| budget:", BUDGET_HOURS, "h")

stage: smoke s0  | budget: 10.5 h


## 2. Pull the code (public repo)

Cloned into `/kaggle/temp` so every session gets fresh code and the 20 GB output
quota is spent only on results.

In [14]:
import os
import subprocess
import shutil
from pathlib import Path

CODE = Path("/kaggle/working/Continual-RL-Project")

# Make sure we're not inside a directory we're about to delete
os.chdir("/kaggle/working")

# Remove old clone if it exists
if CODE.exists():
    shutil.rmtree(CODE)

# Clone repository
subprocess.run(
    ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(CODE)],
    check=True
)

# Set project directory
PROJ = CODE / REPO_SUBDIR if REPO_SUBDIR else CODE

assert (PROJ / "run_kaggle.sh").exists(), (
    f"run_kaggle.sh not in {PROJ} - check REPO_SUBDIR"
)

os.chdir(PROJ)

print("code at:", PROJ)
print(
    subprocess.run(
        ["git", "log", "-1", "--oneline"],
        cwd=CODE,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()
)

Cloning into '/kaggle/working/Continual-RL-Project'...


code at: /kaggle/working/Continual-RL-Project/metaworld
73e6300 merge on new task config


## 3. Resume from the previous session

`Save & Run All` starts from an empty `/kaggle/working`. To continue a run:
`Add-ons → Add data → Your Work / Notebook Output` → pick this notebook's last
version. It mounts read-only under `/kaggle/input/`; this cell copies it back so
the run manifests are found and finished work is skipped.

No-op on the first run.

In [15]:
import glob

restored = 0
for src_root in sorted(glob.glob("/kaggle/input/*")):
    for name in ("runs", "agents", "analysis", "analysis_scratch",
                 "scratch_models", "pretrained_encoders", "plots", "logs"):
        src = pathlib.Path(src_root) / name
        if not src.is_dir():
            continue
        shutil.copytree(src, WORK / name, dirs_exist_ok=True)
        n = sum(1 for _ in src.rglob("*"))
        restored += n
        print(f"restored {name} from {src_root} ({n} entries)")
print("nothing to restore - first run" if restored == 0 else f"restored {restored} entries")

nothing to restore - first run


## 4. Install

Meta-World is installed at a **pinned commit** with `--no-deps`, so pip never
resolves its stale dependency pins and never downgrades torch / numpy /
gymnasium underneath the run. MuJoCo goes in first; everything else comes from
`requirements.txt` with the `metaworld`/`mujoco` lines filtered out.

`MUJOCO_GL=egl` avoids the OpenGL context error some images raise on env
construction even when nothing is rendered.

In [16]:
os.environ["MUJOCO_GL"] = "egl"          # try "osmesa" if EGL errors appear
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["KAGGLE_WORKING"] = str(WORK)

!bash run_kaggle.sh setup

>>> setup
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 65.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 13.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
metaworld OK: /usr/local/lib/python3.12/dist-packages/metaworld/__init__.py
torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4 | count: 2
real CUDA kernel test: OK 0.036871287971735
mw_easy4  (sequence: (0, 2, 3, 1, 0, 3, 2, 1))
  0: window-close-v2            [easy]  CW triplet #3 source; documented positive transfer to peg-unplug-side
  1:

## 5. Sanity check (no GPU time)

Builds every task and asserts constant obs/action shapes plus the
`success` / `task_error` info keys. If this fails, stop — nothing downstream
will be meaningful.

In [17]:
!bash run_kaggle.sh sanity

>>> sanity (no GPU time)

=== chain fusion=classic_cka distillation=False alpha_mass=False ===
2026-09-03 19:14:36.030 | INFO     | cka_rl:__init__:135 - shared alpha: None
2026-09-03 19:14:36.031 | INFO     | cka_rl:__init__:181 - Initializing root shared encoder from scratch
  root base + v1 seeded OK
2026-09-03 19:14:39.786 | INFO     | cka_rl:__init__:135 - shared alpha: Parameter containing:
tensor([0.0010], requires_grad=True)
2026-09-03 19:14:39.787 | INFO     | cka_rl:__init__:173 - Loading frozen encoder from base /tmp/cka_pool_sanity/classic_cka_False_False/task0
2026-09-03 19:14:39.788 | INFO     | cka_rl:__init__:197 - Shared encoder frozen
  shared alpha + frozen encoder + no-merge finalize OK
2026-09-03 19:14:39.831 | INFO     | cka_rl:__init__:135 - shared alpha: Parameter containing:
tensor([-0.0168, -0.0902], requires_grad=True)
2026-09-03 19:14:39.831 | INFO     | cka_rl:__init__:173 - Loading frozen encoder from base /tmp/cka_pool_sanity/classic_cka_False_False/task0

## 6. Run the stage

`timeout` stops the work before the wall so the version still commits.
**Exit code 124 means "budget reached", not an error** — the next session
resumes from the manifests.

In [18]:
import time

secs = int(BUDGET_HOURS * 3600)
cmd = f"timeout --signal=INT {secs} bash run_kaggle.sh {STAGE} {ARG1} {ARG2}".strip()
print(cmd, flush=True)

t0 = time.time()
rc = subprocess.run(cmd, shell=True).returncode
elapsed = (time.time() - t0) / 3600

print(f"\nstage={STAGE} {ARG1} {ARG2} rc={rc} elapsed={elapsed:.2f} h")
if rc == 124:
    print("BUDGET REACHED - not a failure. Finished work is checkpointed. "
          "Commit this version, add its output as input, run the same stage again.")
elif rc != 0:
    raise SystemExit(f"stage failed with code {rc} - see the log above")
else:
    print("stage complete.")

timeout --signal=INT 37800 bash run_kaggle.sh smoke s0
>>> smoke: full pipeline, tiny budget
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
2026-09-03 19:15:54.956 | INFO     | 

## 7. What was produced

In [22]:
print(subprocess.run(f"du -sh {WORK}/* 2>/dev/null | sort -h", shell=True,
                     capture_output=True, text=True).stdout)
print(subprocess.run(f"du -sh {WORK}", shell=True, capture_output=True, text=True).stdout)
print("\nNext: Save Version -> Save & Run All (Commit), then add THIS version's "
      "output as an input dataset before the next stage.")

4.0K	/kaggle/working/pretrained_encoders
440K	/kaggle/working/logs
492K	/kaggle/working/runs
3.4M	/kaggle/working/scratch_models
5.4M	/kaggle/working/plots
14M	/kaggle/working/analysis_scratch
17M	/kaggle/working/Continual-RL-Project
26M	/kaggle/working/agents
96M	/kaggle/working/analysis

161M	/kaggle/working


Next: Save Version -> Save & Run All (Commit), then add THIS version's output as an input dataset before the next stage.
